In [30]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

print("Libraries loaded ✅")

Libraries loaded ✅


###  load  the  data sets   


In [31]:
df = pd.read_csv("../../data/processed/churn_cleaned.csv")
print(f"Shape: {df.shape}")
df.head()   

Shape: (7043, 34)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,...,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0.0,1,29.85,29.85,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,1.0,0,56.95,1889.50,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1,0,0,0,2,1,0.0,1,53.85,108.15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,0,0,45,0,1.0,0,42.30,1840.75,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,0,0,2,1,0.0,1,70.70,151.65,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### select   the X and  Y   

In [32]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print(f"Features: {X.shape[1]}")
print(f"Churn rate: {y.mean():.2%}")

Features: 33
Churn rate: 26.54%


###  split  the  data  into train and  test  data 

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}")

Train: 5634 | Test: 1409
Train churn rate: 26.54% | Test churn rate: 26.54%


###  sacle  numarical  featuers  for  the  logistic  regression   

In [34]:
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlySpend", "TotalServices"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Scaled ✅")

Scaled ✅


In [35]:
print(y_train.value_counts())
print(f"\nImbalance ratio: {(y_train.value_counts()[0] / y_train.value_counts()[1]):.2f} : 1")

Churn
0    4139
1    1495
Name: count, dtype: int64

Imbalance ratio: 2.77 : 1


### smote  

In [36]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
X_train_scaled_smote, y_train_scaled_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_smote.value_counts().to_dict())

Before SMOTE: {0: 4139, 1: 1495}
After SMOTE: {0: 4139, 1: 4139}


In [37]:
def evaluate(name, y_true, y_pred, y_proba):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba)
    }

all_results = []

###  logistic regression  class weight   

In [38]:
log_reg_cw = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg_cw.fit(X_train_scaled, y_train)

pred = log_reg_cw.predict(X_test_scaled)
proba = log_reg_cw.predict_proba(X_test_scaled)[:, 1]
all_results.append(evaluate("Logistic Regression (class_weight)", y_test, pred, proba))

print("✅ Logistic Regression (class_weight) trained")

✅ Logistic Regression (class_weight) trained


### logistic regression smote    

In [39]:
log_reg_smote = LogisticRegression(max_iter=1000, random_state=42)
log_reg_smote.fit(X_train_scaled_smote, y_train_scaled_smote)

pred = log_reg_smote.predict(X_test_scaled)
proba = log_reg_smote.predict_proba(X_test_scaled)[:, 1]
all_results.append(evaluate("Logistic Regression (SMOTE)", y_test, pred, proba))

print("✅ Logistic Regression (SMOTE) trained")

✅ Logistic Regression (SMOTE) trained


###  RandomForestClassifier  class  weight    

In [40]:
rf_cw = RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf_cw.fit(X_train, y_train)

pred = rf_cw.predict(X_test)
proba = rf_cw.predict_proba(X_test)[:, 1]
all_results.append(evaluate("Random Forest (class_weight)", y_test, pred, proba))

print("✅ Random Forest (class_weight) trained")

✅ Random Forest (class_weight) trained


###  RandomForestClassifier  smote    

In [41]:
rf_smote = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)

pred = rf_smote.predict(X_test)
proba = rf_smote.predict_proba(X_test)[:, 1]
all_results.append(evaluate("Random Forest (SMOTE)", y_test, pred, proba))

print("✅ Random Forest (SMOTE) trained")

✅ Random Forest (SMOTE) trained


### XGBClassifier  class  weight    

In [42]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_cw = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric="logloss"
)
xgb_cw.fit(X_train, y_train)

pred = xgb_cw.predict(X_test)
proba = xgb_cw.predict_proba(X_test)[:, 1]
all_results.append(evaluate("XGBoost (scale_pos_weight)", y_test, pred, proba))

print("✅ XGBoost (scale_pos_weight) trained")

✅ XGBoost (scale_pos_weight) trained


###  XGBClassifier smote  

In [43]:
xgb_smote = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    random_state=42, eval_metric="logloss"
)
xgb_smote.fit(X_train_smote, y_train_smote)

pred = xgb_smote.predict(X_test)
proba = xgb_smote.predict_proba(X_test)[:, 1]
all_results.append(evaluate("XGBoost (SMOTE)", y_test, pred, proba))

print("✅ XGBoost (SMOTE) trained")

✅ XGBoost (SMOTE) trained


###  evaluate  the  all  models    

In [44]:
results = pd.DataFrame(all_results).set_index("Model").round(3)
results.sort_values("Recall", ascending=False)

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Logistic Regression (class_weight),0.737,0.503,0.786,0.613,0.841
XGBoost (scale_pos_weight),0.761,0.535,0.765,0.629,0.839
Logistic Regression (SMOTE),0.731,0.496,0.749,0.596,0.831
Random Forest (class_weight),0.776,0.560,0.735,0.636,0.840
Random Forest (SMOTE),0.775,0.567,0.647,0.604,0.836
XGBoost (SMOTE),0.786,0.603,0.572,0.587,0.839


###  get   the  best   model   

In [45]:
scores = cross_val_score(rf_cw, X_train, y_train, cv=5, scoring="roc_auc")
print(f"Random Forest (class_weight): ROC-AUC = {scores.mean():.3f} (+/- {scores.std():.3f})")

Random Forest (class_weight): ROC-AUC = 0.842 (+/- 0.014)


In [46]:
# Business reasoning:
# - ROC-AUC nearly identical across all models tested → signal quality is comparable regardless of algorithm
# - Recall matters most for churn (missing a churner is costly), but very low precision wastes retention budget
# - Random Forest (class_weight) gives the best F1 balance (0.636), strong recall (0.735),
#   competitive ROC-AUC (0.840), and interpretable feature_importances_ for the retention engine
# - Also compared against Label Encoding across all 3 algorithms — One-Hot/Ordinal/Binary
#   consistently performed better (see note below), so that approach was kept as final

best_model_name = "Random Forest (class_weight, One-Hot/Ordinal/Binary)"
best_model = rf_cw

print(f"Selected model: {best_model_name}")

Selected model: Random Forest (class_weight, One-Hot/Ordinal/Binary)


###  save  the  model   

In [47]:
joblib.dump(best_model, "../models/churn_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

print("Model and scaler saved ✅")

Model and scaler saved ✅


In [48]:
results.reset_index().to_json("../models/model_comparison.json", orient="records", indent=2)
print("Model comparison saved ✅")

Model comparison saved ✅


## Model Training Summary
- Trained 3 algorithms × 2 imbalance strategies (class_weight/scale_pos_weight vs SMOTE) = 6 models
- ROC-AUC nearly identical across all models (0.831–0.841) → algorithm choice matters less than threshold/imbalance strategy
- **Key finding:** SMOTE underperformed algorithm-level class weighting on Recall across all algorithms —
  likely due to unrealistic synthetic samples generated from interpolating one-hot encoded categorical features
- **Selected model: Random Forest (class_weight)** — best F1 (0.636), strong recall (0.735)
  with acceptable precision (0.560), interpretable feature importances

## Encoding Comparison Note
Also tested Label Encoding as an alternative to One-Hot/Ordinal/Binary encoding across all 3 algorithms.
Result: One-Hot/Ordinal/Binary consistently outperformed Label Encoding, particularly for Random Forest
(Recall: 0.735 vs 0.701, F1: 0.636 vs 0.619). Label encoding was dropped in favor of the current approach.

- Saved: `churn_model.pkl`, `scaler.pkl`, `model_comparison.json`